In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import ollama
import json
import pandas as pd

from iterative_eval_utils import (
    compute_bertscore_batched,
    prepare_final_eval_df,
    run_resumable_evaluation,
)
from retrieval_guardrails import DEFAULT_POLICY, is_allowed_document


In [3]:
# Load QULAC dataset from JSON file
# file contains queries, clarification questions, and metadata
with open(r"C:\Raina's Files\Semester IV - Thesis\Experiments\qulac.json") as f:
    data = json.load(f)

# convert to rows
df = pd.DataFrame(data)


In [4]:
# print column names
print(df.columns)

Index(['topic_id', 'facet_id', 'topic_facet_id', 'topic_facet_question_id',
       'topic', 'topic_type', 'facet_type', 'topic_desc', 'facet_desc',
       'question', 'answer'],
      dtype='object')


In [5]:
# print first row
df.iloc[0]

topic_id                                                                   1
facet_id                                                                   1
topic_facet_id                                                           1-1
topic_facet_question_id                                                1-1-1
topic                                                      obama family tree
topic_type                                                           faceted
facet_type                                                               nav
topic_desc                 Find information on President Barack Obama\'s ...
facet_desc                 Find the TIME magazine photo essay \"Barack Ob...
question                   are you interested in seeing barack obamas family
answer                                    yes am interested in obamas family
Name: 0, dtype: object

In [6]:
# unique queries
unique_queries = df[['topic_id', 'topic']].drop_duplicates()

sample_queries = unique_queries.sample(n=30, random_state=42)

sample_df = df[df['topic_id'].isin(sample_queries['topic_id'])]

# Reset index
sample_df = sample_df.reset_index(drop=True)
sample_df['ID'] = sample_df.index + 1

sample_df = sample_df[['ID', 'topic_id', 'topic', 'question', 'answer']]

pd.set_option('display.max_colwidth', None)
print(sample_df.head(10))

   ID  topic_id                topic  \
0   1       108  ralph owen brewster   
1   2       108  ralph owen brewster   
2   3       108  ralph owen brewster   
3   4       108  ralph owen brewster   
4   5       108  ralph owen brewster   
5   6       108  ralph owen brewster   
6   7       108  ralph owen brewster   
7   8       108  ralph owen brewster   
8   9       108  ralph owen brewster   
9  10       108  ralph owen brewster   

                                                                                                                                             question  \
0                                                                                      are you interested in learning more about ralph owen brewseter   
1                                                                             would you like to know the political positions ralph owen brewster held   
2  would you like to know what movie star portrayed brewster in the movie the aviator and what award

In [7]:
# drop empty question
sample_df = sample_df[sample_df['question'].notna()]
sample_df = sample_df[sample_df['question'].str.strip() != ""]

In [8]:
def is_valid_answer(x):
    x = str(x).lower().strip()
    return len(x) > 15 and x not in ["yes", "no", "i dont know"]

sample_df = df[df['answer'].apply(is_valid_answer)]

In [9]:
# get wikipedia docs for a query
import wikipedia

def get_wiki_docs(query, top_k=5, policy=DEFAULT_POLICY):
    docs = []

    try:
        titles = wikipedia.search(query, results=top_k * 3)

        for title in titles:
            try:
                page = wikipedia.page(title, auto_suggest=False)
                if not is_allowed_document(page.title, page.summary, policy=policy):
                    continue
                docs.append({
                    "title": page.title,
                    "summary": page.summary
                })
                if len(docs) >= top_k:
                    break
            except:
                continue

    except:
        pass

    return docs


In [10]:
test_queries = sample_df['topic'].drop_duplicates().sample(n=10, random_state=42).tolist()

for i, query in enumerate(test_queries, 1):
    raw_docs = get_wiki_docs(query)
    print(f"{i}. {query}")
    print([doc["title"] for doc in raw_docs])
    print("------")

1. porterville
['Porterville, California', 'Tulare County, California', 'Porterville, Texas', 'Porterville College', 'Porterville Transit']
------
2. rick warren
['Saddleback Church', 'Kay Warren (author)', 'Sam Harris', 'Daniel Amen', 'The Purpose Driven Life']
------
3. adobe indian houses
['Leonis Adobe', 'Second Battle of Adobe Walls', 'Adobe Walls, Texas', 'First Battle of Adobe Walls', 'Salvador Vallejo Adobe']
------
4. indexed annuity
['Annuity', 'Equity-indexed annuity', 'Fixed annuity', 'Annuities in the United States', 'F&G']
------
5. bellevue
['Bellevue, Washington', 'Bellevue Hospital', 'Bellevue (Stockholm)', '2 Line (Sound Transit)', 'Bellevue, Maryland']
------
6. elliptical trainer
['Elliptical trainer', 'Arc Trainer', 'ElliptiGO', 'Stationary bicycle', 'List of exercise equipment']
------
7. alexian brothers hospital
['Alexians', 'St. Alexius Hospital (Missouri)', 'John Dillinger', 'Good Samaritan Hospital (San Jose)', "Saint Valentine's Day Massacre"]
------
8. dutc

In [11]:
# compute once per unique topic
topic_to_docs = {
    topic: get_wiki_docs(topic)
    for topic in sample_df['topic'].unique()
}

# map back
sample_df['retrieved_docs'] = sample_df['topic'].map(topic_to_docs)

pd.set_option('display.max_colwidth', None)
print(sample_df[['topic', 'retrieved_docs']].head(3))

               topic  \
0  obama family tree   
1  obama family tree   
2  obama family tree   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [12]:
import re
def clean_query_for_retrieval(query):
    query = query.lower()

    # extract text inside quotes (strong signal)
    match = re.search(r"'([^']+)'", query)
    if match:
        return match.group(1)

    # fallback: remove common question patterns
    query = re.sub(r"what specific information about", "", query)
    query = re.sub(r"can you clarify what you mean by", "", query)
    query = re.sub(r"are you looking for", "", query)

    # remove punctuation
    query = re.sub(r"[^a-z\s]", "", query)

    return query.strip()

In [13]:
def filter_relevant_docs(query, docs, policy=DEFAULT_POLICY):
    import re

    def normalize(text):
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    query_norm = normalize(query)
    query_words = [
        w for w in query_norm.split()
        if w not in {"the", "and", "of", "to", "in", "for", "on", "a", "an"}
        and len(w) > 2
    ]

    generic_query = len(query_words) <= 2
    scored_docs = []

    for doc in docs:
        if isinstance(doc, dict):
            title = doc.get("title", "")
            summary = doc.get("summary", "")
        else:
            title = ""
            summary = str(doc)

        if not is_allowed_document(title, summary, policy=policy):
            continue

        title_norm = normalize(title)
        doc_norm = normalize(summary)

        lexical_score = sum(1 for w in query_words if w in doc_norm)
        title_score = sum(2 for w in query_words if w in title_norm)
        score = lexical_score + title_score

        if generic_query and (" in " in title_norm or " list of " in title_norm):
            score -= 2

        prompt = f"""
        Query: {query}

        Document title:
        {title}

        Document:
        {summary[:1200]}

        Task:
        Is this document directly relevant to answering the query?

        Rules:
        - Prefer documents about the exact entity or topic in the query.
        - For generic queries, prefer broad topic pages over location-specific pages.
        - Reject documents that only match one common word.
        - Reject policy-disallowed or academically unsuitable documents.
        - Answer YES only if the document is clearly about the same topic.

        Answer ONLY:
        YES or NO
        """

        response = ollama.chat(
            model='llama3.2:3b',
            messages=[{'role': 'user', 'content': prompt}],
            options={'num_predict': 5, 'temperature': 0}
        )

        ans = response['message']['content'].strip().upper()

        if "YES" in ans and score > 0:
            scored_docs.append((score, summary))

    scored_docs = sorted(scored_docs, key=lambda x: x[0], reverse=True)
    filtered = [doc for _, doc in scored_docs[:3]]

    if filtered:
        return filtered

    lexical_fallback = []
    for doc in docs:
        if isinstance(doc, dict):
            title = doc.get("title", "")
            summary = doc.get("summary", "")
        else:
            title = ""
            summary = str(doc)

        if not is_allowed_document(title, summary, policy=policy):
            continue

        title_norm = normalize(title)
        doc_norm = normalize(summary)
        score = sum(1 for w in query_words if w in doc_norm) + sum(2 for w in query_words if w in title_norm)
        if generic_query and (" in " in title_norm or " list of " in title_norm):
            score -= 2
        if score > 0:
            lexical_fallback.append((score, summary))

    lexical_fallback = sorted(lexical_fallback, key=lambda x: x[0], reverse=True)
    if lexical_fallback:
        return [doc for _, doc in lexical_fallback[:2]]

    if docs:
        first_doc = docs[0]
        if isinstance(first_doc, dict) and is_allowed_document(first_doc.get("title", ""), first_doc.get("summary", ""), policy=policy):
            return [first_doc.get("summary", "")]
        if not isinstance(first_doc, dict) and is_allowed_document("", str(first_doc), policy=policy):
            return [str(first_doc)]

    return []


In [14]:
# detect ambiguity BEFORE answer generation
def detect_ambiguity(query):
    query = query.strip()

    # well-formed named entities usually do not need intent clarification
    if len(query.split()) >= 3 and query.replace(" ", "").isalpha():
        return False

    prompt = f"""
    Query: {query}

    Does this query have multiple meanings or interpretations,
    or is it missing the intent needed to answer it correctly?

    Answer only: YES or NO
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 5, 'temperature': 0}
    )

    return "YES" in response['message']['content'].upper()


In [15]:
# Generate answer
def generate_answer(question, docs, original_query=None):
    import re

    if not docs:
        return "Insufficient information"

    topic = original_query or question
    context = "\n\n".join([f"Doc{i+1}: {doc[:900]}" for i, doc in enumerate(docs[:2])])

    prompt = f"""
    You are answering a question using ONLY the provided context.

    Topic:
    {topic}

    Question to answer:
    {question}

    STRICT RULES:
    - Use ONLY facts from the context.
    - Keep the answer anchored to the topic.
    - Ignore unrelated facts even if they appear in the context.
    - Do NOT combine unrelated entities or topics.
    - If the context does not support the answer, say: "Insufficient information".
    - If the topic is ambiguous and the context does not resolve it, say: "Ambiguous query".

    RESPONSE RULES:
    - Write 1 or 2 complete sentences only.
    - Answer the question directly.
    - Do not trail off.
    - Do not guess.
    - Do not contradict the context.

    Context:
    {context}

    Answer:
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={
            'num_predict': 70,
            'temperature': 0
        }
    )

    answer = response['message'].get('content', '').strip()
    answer = re.sub(r'\s+', ' ', answer).strip()

    bad_patterns = [
        "user might be looking",
        "the query",
        "it can be inferred",
        "this suggests",
        "likely",
        "generally",
        "typically",
        "in most cases",
        "it seems"
    ]

    if len(answer.split()) < 5:
        return "Insufficient information"

    if any(pattern in answer.lower() for pattern in bad_patterns):
        return "Insufficient information"

    if not answer.endswith(('.', '!', '?')):
        answer += '.'

    sentences = re.split(r'(?<=[.!?])\s+', answer)
    cleaned = []
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        cleaned.append(sentence)
        if len(cleaned) == 2:
            break

    final_answer = ' '.join(cleaned).strip()

    if not final_answer or len(final_answer.split()) < 5:
        return "Insufficient information"

    return final_answer


In [16]:
# get 3 unique queries
unique_df = sample_df.drop_duplicates(subset=['topic'])

# random sample of 3
test_df = unique_df.sample(n=3, random_state=42).copy()

answers = []

for _, row in test_df.iterrows():
    query = row['topic']

    clean_query = clean_query_for_retrieval(query)

    raw_docs = get_wiki_docs(query)
    filtered_docs = filter_relevant_docs(query, raw_docs)

    ans = generate_answer(query, filtered_docs)
    answers.append(ans)

test_df['answer'] = answers

In [17]:
from IPython.display import display
pd.set_option('display.max_colwidth', None)
display(test_df[['topic', 'answer']])

,topic,answer
3262,porterville,"Insufficient information. The context provides information about Porterville, California, but not about Porterville, Texas."
6040,rick warren,"Rick Warren is a Christian pastor who wrote the Bible study book ""The Purpose Driven Life""."
865,adobe indian houses,"Adobe Indian houses are not specifically mentioned in the context as a distinct architectural style or type of dwelling associated with Native American communities. However, the context does mention that Adobe Walls trading post had local Native American trade in its vicinity, suggesting some form of traditional or cultural dwellings may have existed nearby."


In [18]:
# Baseline follow-up generation (plain prompting)

def ensure_question(text, query):
    text = str(text).strip()
    text = text.replace("Question:", "").replace("Follow-up", "").replace(":", "").strip()
    text = " ".join(text.split())

    if not text:
        return f"What specific information are you looking for about {query}?"

    if "?" in text:
        text = text.split("?")[0].strip() + "?"
        return text

    lowered = text.lower()
    if lowered.startswith(("what ", "which ", "who ", "where ", "when ", "why ", "how ", "are ", "is ", "do ", "does ", "did ", "can ", "could ", "would ", "should ")):
        return text.rstrip(".!") + "?"

    if lowered.endswith(" is") or lowered.endswith(" are"):
        return text.rstrip(".!") + "?"

    return f"What specific information are you looking for about {query}?"


def generate_baseline_followup(query):
    prompt = f"""
    Generate ONE useful follow-up question to clarify the query.

    Rules:
    - Ask about a specific missing detail (e.g., location, type, features, time, etc.)
    - Do NOT ask vague questions like "What do you want to know?"
    - The question must narrow down the query and be relevant to the original query and potential information needs.
    - Frame the output as a question.
    - The output must end with a question mark.
    - Do not answer the query.
    - Do not complete the sentence as a statement.

    Query: {query}

    Return ONLY the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    msg = response['message']
    text = msg.get('content', '').strip()

    return ensure_question(text, query)


In [19]:
# test
print(generate_baseline_followup("flushing"))
print(generate_baseline_followup("ontario california airport"))
print(generate_baseline_followup("ralph owen brewster"))
print(generate_baseline_followup("cheap internet"))

Is there a specific context or industry (e.g. plumbing, environmental, etc.) where you would like to know more about flushing?
Is Ontario, California referring to the Ontario International Airport (ONT) specifically?
Was Ralph Owen Brewster an American inventor or engineer?
Is there a specific region or country where you're looking for affordable internet options?


In [20]:
# Aspect extraction
def extract_aspects(query):
    import re

    query_clean = str(query).strip()
    query_lower = query_clean.lower()
    words = [w for w in re.findall(r"[a-zA-Z0-9]+", query_clean) if len(w) > 1]

    def has_any(phrases):
        return any(p in query_lower for p in phrases)

    def classify_query_type(text):
        text_lower = text.lower().strip()
        tokens = [w for w in re.findall(r"[a-zA-Z0-9]+", text_lower) if len(w) > 1]

        if len(tokens) <= 2:
            return "ambiguous_short_query"

        if has_any(["family tree", "ancestry", "genealogy"]):
            return "family_relationship"

        if has_any(["hotel", "resort", "inn", "airport", "terminal", "airlines"]):
            return "place_or_facility"

        if has_any(["restaurant", "museum", "university", "company", "organization", "school", "farm"]):
            return "organization_or_place"

        if has_any(["cheap", "best", "budget", "affordable", "price", "pricing", "cost", "provider", "plan"]):
            return "service_or_product"

        if has_any(["map", "located", "location", "address", "directions", "where is"]):
            return "place_or_location"

        if has_any(["who won", "biography", "born", "died", "who is", "who was"]):
            return "person"

        # Broad person-name heuristic without memorizing topic-specific exceptions.
        title_cased = [part for part in re.split(r"\s+", text.strip()) if part]
        if 2 <= len(title_cased) <= 4 and all(part[:1].isalpha() for part in title_cased):
            non_person_markers = {
                "airport", "hotel", "resort", "museum", "restaurant", "company",
                "university", "school", "farm", "lake", "park", "internet",
                "family", "tree", "map", "disease", "county", "tourism"
            }
            if not any(token in non_person_markers for token in tokens):
                return "person"

        if has_any(["disease", "symptoms", "treatment", "causes"]):
            return "topic_or_condition"

        if has_any(["album", "movie", "film", "book", "song", "game"]):
            return "work_or_media"

        return "generic"

    query_type = classify_query_type(query_clean)

    aspect_priors = {
        "family_relationship": ["family members", "ancestry", "relationships"],
        "service_or_product": ["type or option", "pricing or availability", "features or providers"],
        "place_or_facility": ["location", "services or amenities", "current status"],
        "place_or_location": ["location", "key features", "relevance or use"],
        "organization_or_place": ["main purpose or type", "location", "notable features"],
        "person": ["identity", "role or career", "notable contribution"],
        "topic_or_condition": ["definition or type", "causes or features", "diagnosis or treatment"],
        "work_or_media": ["work identity", "content or subject", "notable significance"],
        "ambiguous_short_query": ["intended meaning", "context", "specific need"],
    }

    if query_type in aspect_priors:
        return aspect_priors[query_type]

    prompt = f"""
    Identify 3 key aspects needed to answer this query well.

    An aspect is a dimension of information required to satisfy the query,
    including implicit missing information when relevant.

    Rules:
    - exactly 3 aspects
    - short phrases only
    - 2 to 5 words each
    - no full sentences
    - avoid generic items like "details", "information", "major achievements", or "historical significance"
    - prefer aspects that help generate useful follow-up questions for this specific query type
    - if the query is about a person, use person-style aspects only when appropriate
    - if the query is about a place, organization, service, or object, do NOT use person-style aspects

    Query: {query_clean}

    Aspects:
    """
    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 80, 'temperature': 0}
    )

    def clean_aspects(text):
        lines = text.split("\n")
        cleaned = []
        banned = {
            "details", "information", "general information", "overview", "background",
            "major achievements", "historical significance"
        }

        for line in lines:
            line = line.strip()
            line = line.lstrip("1234567890.-) ").strip()
            line = re.sub(r'[^a-zA-Z0-9\s/&-]', '', line).strip()

            if not line:
                continue
            if len(line.split()) < 1 or len(line.split()) > 6:
                continue
            if line.lower() in banned:
                continue
            if any(x in line.lower() for x in ["part", "definition", "usage"]):
                continue

            cleaned.append(line)

        deduped = []
        seen = set()
        for item in cleaned:
            key = item.lower()
            if key not in seen:
                seen.add(key)
                deduped.append(item)

        return deduped[:3]

    raw = response['message']['content']
    aspects = clean_aspects(raw)

    if len(aspects) < 3:
        fallback = ["main topic", "key facts", "context"]
        for item in fallback:
            if len(aspects) < 3 and item.lower() not in {a.lower() for a in aspects}:
                aspects.append(item)

    return aspects[:3]


In [21]:
# test aspect extraction
test_queries = sample_df['topic'].drop_duplicates().sample(n=10, random_state=42).tolist()

for i, query in enumerate(test_queries, 1):
    print(f"{i}. {query}")
    print(extract_aspects(query))
    print("------")

1. porterville
['intended meaning', 'context', 'specific need']
------
2. rick warren
['intended meaning', 'context', 'specific need']
------
3. adobe indian houses
['identity', 'role or career', 'notable contribution']
------
4. indexed annuity
['intended meaning', 'context', 'specific need']
------
5. bellevue
['intended meaning', 'context', 'specific need']
------
6. elliptical trainer
['intended meaning', 'context', 'specific need']
------
7. alexian brothers hospital
['identity', 'role or career', 'notable contribution']
------
8. dutchess county tourism
['Nearby Attractions', 'Seasonal Events', 'Local Cuisine']
------
9. california franchise tax board
['identity', 'role or career', 'notable contribution']
------
10. to be or not to be that is the question
['Contextual setting', 'Philosophical framework', 'Authors intent']
------


In [22]:
# Missing aspect identification
def find_missing_aspect(query, answer, aspects, allow_intent=False):
    import re

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    answer_norm = normalize(answer)

    if allow_intent and detect_ambiguity(query):
        return "INTENT"

    weak_answer = (
        not answer_norm
        or "insufficient information" in answer_norm
        or "ambiguous query" in answer_norm
        or len(answer_norm.split()) < 12
        or not str(answer).strip().endswith((".", "?", "!"))
    )

    if not aspects:
        return "DETAIL" if weak_answer else "NONE"

    stopwords = {"and", "of", "the", "to", "from", "for", "in", "on", "with", "or"}
    coverage = {}
    uncovered = []

    for aspect in aspects:
        aspect_norm = normalize(aspect)
        aspect_words = [w for w in aspect_norm.split() if w not in stopwords and len(w) > 2]

        if not aspect_words:
            coverage[aspect] = 0.0
            uncovered.append(aspect)
            continue

        matched = sum(1 for w in aspect_words if w in answer_norm)
        ratio = matched / max(len(aspect_words), 1)
        coverage[aspect] = ratio

        if ratio < 0.6:
            uncovered.append(aspect)

    if not uncovered:
        return "DETAIL" if weak_answer else "NONE"

    if weak_answer and len(uncovered) == len(aspects):
        return uncovered[0]

    prompt = f"""
    Original query: {query}

    Combined answer so far: {answer}

    Candidate aspects and approximate coverage:
    {coverage}

    Remaining candidate aspects:
    {uncovered}

    Choose the ONE most important aspect still missing to answer the original query well.

    Rules:
    - Return NONE only if the answer is already sufficient.
    - Otherwise return exactly one aspect from the remaining candidate list.
    - Prefer the aspect whose absence most limits answering the original query.
    - Do not invent a new aspect.

    Output format:
    Missing: <NONE or exact aspect>
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 30, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "Missing:" in text:
        missing = text.split("Missing:")[1].split("\n")[0].strip()
    else:
        missing = uncovered[0]

    if missing == "NONE":
        return "DETAIL" if weak_answer else "NONE"

    if missing in uncovered:
        return missing

    return uncovered[0]


In [23]:
# test to get answer
query = "ralph owen brewster"

raw_docs = get_wiki_docs(query)
filtered_docs = filter_relevant_docs(query, raw_docs)

answer = generate_answer(query, filtered_docs)

print("Query:", query)
print("Answer:", answer)

Query: ralph owen brewster
Answer: Ralph Owen Brewster was an American politician from Maine who served as the 54th governor of Maine from 1925 to 1929. He also held offices in the U.S.


In [25]:
# test missing aspect identification

query = "ralph owen brewster"
answer = answer

aspects = [
    "British sociologist and political economist", "Founder of the Brewster Corporation", "Author of \"Social Stratification\""
]

missing = find_missing_aspect(query, answer, aspects)

print("Query:", query)
print("Answer:", answer)
print("Aspects:", aspects)
print("Missing:", missing)

Query: ralph owen brewster
Answer: Ralph Owen Brewster was an American politician who served as the 54th governor of Maine from 1925 to 1929 and in the U.S. House of Representatives from 1935 to 1941. He was a close confidant of Joseph McCarthy and an antagonist of Howard Hughes. Brewster was defeated by Frederick G. Payne
Aspects: ['British sociologist and political economist', 'Founder of the Brewster Corporation', 'Author of "Social Stratification"']
Missing: British sociologist and political economist


In [26]:
# Improved follow-up generation

def generate_aspect_followup(query, missing_aspect, aspects=None, asked_followups=None):
    asked_followups = asked_followups or []
    asked_norm = {q.strip().lower() for q in asked_followups if q}

    if missing_aspect == "NONE":
        return None

    if missing_aspect == "INTENT":
        candidate = f"What specific information do you want to know about {query}?"
        return None if candidate.lower() in asked_norm else candidate

    if missing_aspect == "DETAIL":
        candidate = f"What specific detail about {query} are you looking for?"
        return None if candidate.lower() in asked_norm else candidate

    prompt = f"""
    Original query: {query}

    Missing aspect: {missing_aspect}

    Generate ONE clear, direct clarification question asking only about this missing aspect.

    Rules:
    - Start with What, Which, Who, Where, or When.
    - Keep it specific and factual.
    - Mention the original topic naturally.
    - Do not ask a generic question.
    - Do not repeat the aspect wording awkwardly.
    - Output only the question.
    """

    response = ollama.chat(
        model='llama3.2:3b',
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': 50, 'temperature': 0}
    )

    text = response['message']['content'].strip()
    if "?" in text:
        text = text.split("?")[0].strip() + "?"
    elif text:
        text = text.strip() + "?"

    if not text:
        text = f"What about {missing_aspect} is important for {query}?"

    if text.lower() in asked_norm:
        simple = f"What is {missing_aspect} for {query}?"
        if simple.lower() not in asked_norm:
            return simple
        return None

    return text


In [27]:
# test improved follow-up question generation
query = "ralph owen brewster"

clean_query = clean_query_for_retrieval(query)

docs = get_wiki_docs(query)

docs = filter_relevant_docs(query, docs)

ans = generate_answer(query, docs)

aspects = extract_aspects(query)
missing = find_missing_aspect(query, ans, aspects)
imp_q = generate_aspect_followup(query, missing, aspects)

print("Aspects:", aspects)
print("Missing aspect:", missing)
print("Answer:", ans)
print("Improved:", imp_q)


Aspects: ['identity', 'role or career', 'notable contribution']
Missing aspect: role or career
Answer: Ralph Owen Brewster was an American politician from Maine who served as the 54th governor of Maine from 1925 to 1929. He also held offices in the U.S.
Improved: What was Ralph Owen Brewster's profession?


In [28]:
# Iterative pipeline
def run_iterative_pipeline(query, max_steps=3):
    import re

    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    history = []
    base_aspects = extract_aspects(query)
    asked_followups = []
    resolved_aspects = []
    current_query = query

    clean_query = clean_query_for_retrieval(query)
    docs = get_wiki_docs(clean_query)
    docs = filter_relevant_docs(query, docs)

    for step in range(1, max_steps + 1):
        answer = generate_answer(current_query, docs, original_query=query)

        combined_answer_parts = [
            item["answer"]
            for item in history
            if item.get("answer") and item["answer"] != "Insufficient information"
        ]
        if answer and answer != "Insufficient information":
            combined_answer_parts.append(answer)
        combined_answer = " ".join(combined_answer_parts).strip() or answer

        remaining_aspects = [aspect for aspect in base_aspects if aspect not in resolved_aspects]
        missing = find_missing_aspect(
            query,
            combined_answer,
            remaining_aspects,
            allow_intent=(step == 1)
        )

        step_record = {
            "step": step,
            "query": current_query,
            "answer": answer,
            "missing": missing,
            "followup": None
        }

        if missing not in ["NONE", "INTENT", "DETAIL"] and missing in remaining_aspects:
            resolved_aspects.append(missing)

        if missing == "NONE":
            history.append(step_record)
            break

        if current_query != query and answer == "Insufficient information":
            history.append(step_record)
            break

        if step >= max_steps:
            history.append(step_record)
            break

        followup = generate_aspect_followup(
            query,
            missing,
            remaining_aspects,
            asked_followups=asked_followups
        )

        if not followup:
            history.append(step_record)
            break

        if history:
            prev_followup = history[-1].get("followup")
            prev_answer = normalize(history[-1].get("answer", ""))
            curr_answer = normalize(answer)
            combined_prev = normalize(" ".join([
                item["answer"] for item in history
                if item.get("answer") and item["answer"] != "Insufficient information"
            ]))

            no_new_information = curr_answer and curr_answer in combined_prev
            repeated_followup = followup == prev_followup
            highly_similar_answer = curr_answer[:140] == prev_answer[:140] if curr_answer and prev_answer else False

            if repeated_followup or highly_similar_answer or no_new_information:
                history.append(step_record)
                break

        step_record["followup"] = followup
        history.append(step_record)

        asked_followups.append(followup)
        current_query = followup

    return history


In [29]:
# test
result = run_iterative_pipeline("ralph owen brewster")

for step in result:
    print(step)
    print("-----")

{'step': 1, 'query': 'ralph owen brewster', 'answer': 'Ralph Owen Brewster was an American politician from Maine who served as the 54th governor of Maine from 1925 to 1929. He also held offices in the U.S.', 'missing': 'role or career', 'followup': "What was Ralph Owen Brewster's profession?"}
-----
{'step': 2, 'query': "What was Ralph Owen Brewster's profession?", 'answer': 'Ralph Owen Brewster was an American politician who served as governor, member of the U.S. House of Representatives, and U.S.', 'missing': 'notable contribution', 'followup': 'What notable scientific contribution did Ralph Owen Brewster make?'}
-----
{'step': 3, 'query': 'What notable scientific contribution did Ralph Owen Brewster make?', 'answer': 'Insufficient information. The context does not mention any notable scientific contribution made by Ralph Owen Brewster.', 'missing': 'identity', 'followup': None}
-----


In [30]:
# test on multiple queries
results = []

for query in sample_df['topic'].drop_duplicates().head(5):
    res = run_iterative_pipeline(query)
    results.append(res)

In [31]:
for i, res in enumerate(results):
    print(f"Query {i+1}")
    for step in res:
        print(step)
    print("------")

Query 1
{'step': 1, 'query': 'obama family tree', 'answer': "Barack Obama's immediate family includes his wife Michelle Obama and daughters Malia and Sasha. His father is Barack Hussein Obama Sr., who was a Kenyan senior governmental economist.", 'missing': 'ancestry', 'followup': "What is the ancestry of Barack Obama's paternal grandfather?"}
{'step': 2, 'query': "What is the ancestry of Barack Obama's paternal grandfather?", 'answer': "Barack Hussein Obama Sr.'s ancestry is of Luo Kenyan and African-American descent. He also has Old Stock American ancestry, including originally English, Scots-Irish, Welsh, German, and Swiss roots.", 'missing': 'family members', 'followup': "What other family members are included in Barack Obama's immediate family tree?"}
{'step': 3, 'query': "What other family members are included in Barack Obama's immediate family tree?", 'answer': "Barack Obama's immediate family tree includes his wife Michelle Obama and daughters Malia and Sasha. His father is Bar

In [32]:
# use ONE reference per query

def generate_one_step_followup(query):
    docs = get_wiki_docs(query)
    docs = filter_relevant_docs(query, docs)
    answer = generate_answer(query, docs, original_query=query)
    missing = find_missing_aspect(query, answer, extract_aspects(query), allow_intent=True)
    return generate_aspect_followup(query, missing, extract_aspects(query))


def get_iterative_outputs(query):
    history = run_iterative_pipeline(query)
    followups = [step['followup'] for step in history if step.get('followup')]
    first_followup = followups[0] if followups else None
    final_followup = followups[-1] if followups else None
    return {
        'history': history,
        'first_followup': first_followup,
        'final_followup': final_followup,
        'num_steps': len(history),
        'num_followups': len(followups)
    }


test_df = unique_queries.head(5).copy()

test_df['Baseline'] = test_df['topic'].apply(generate_baseline_followup)
test_df['OneStep'] = test_df['topic'].apply(generate_one_step_followup)
test_df['Iterative'] = test_df['topic'].apply(lambda q: get_iterative_outputs(q)['final_followup'])

display(test_df[['topic', 'Baseline', 'OneStep', 'Iterative']])


,topic,Baseline,OneStep,Iterative
0,obama family tree,"Was Barack Obama's father, Barack Obama Sr., Kenyan or American?",What is the ancestry of Barack Obama's paternal grandfather?,What other family members are included in Barack Obama's immediate family tree?
39,cheap internet,Is there a specific region or country where you're looking for affordable internet options?,What specific information do you want to know about cheap internet?,What specific information do you want to know about cheap internet?
129,ritz carlton lake las vegas,"Is the Ritz-Carlton, Lake Las Vegas located within the city limits of Henderson or on the outskirts?","What is the address of The Ritz-Carlton, Lake Las Vegas?","What is the address of The Ritz-Carlton, Lake Las Vegas?"
189,fickle creek farm,Is Fickle Creek Farm located in a specific state or region?,What is the main purpose or type of Fickle Creek Farm?,What is the main purpose or type of Fickle Creek Farm?
234,madam cj walker,Was Madam C.J. Walker an African American entrepreneur who built her hair care business from scratch in the early 20th century?,What was Madam C.J. Walker's profession?,What notable product line did Madam C.J. Walker develop to promote hair care for African American women?


In [35]:
# compute outputs on the full evaluation set with resume support
CHECKPOINT_PATH = "iterative_eval_checkpoint.csv"
SAVE_EVERY = 10

full_df = run_resumable_evaluation(
    unique_queries=unique_queries,
    generate_baseline_followup=generate_baseline_followup,
    generate_one_step_followup=generate_one_step_followup,
    get_iterative_outputs=get_iterative_outputs,
    checkpoint_path=CHECKPOINT_PATH,
    save_every=SAVE_EVERY,
    restart=False,
)

display(full_df.head())


Running evaluation for 198 pending queries out of 198 total.
Evaluation: [#-----------------------------] 10/198 (  5.1%) | elapsed 07:16 | eta 02:16:43Saved checkpoint at 10/198 to iterative_eval_checkpoint.csv.
Evaluation: [###---------------------------] 20/198 ( 10.1%) | elapsed 14:16 | eta 02:07:02Saved checkpoint at 20/198 to iterative_eval_checkpoint.csv.
Evaluation: [####--------------------------] 30/198 ( 15.2%) | elapsed 21:18 | eta 01:59:17Saved checkpoint at 30/198 to iterative_eval_checkpoint.csv.
Evaluation: [######------------------------] 40/198 ( 20.2%) | elapsed 28:58 | eta 01:54:28Saved checkpoint at 40/198 to iterative_eval_checkpoint.csv.
Evaluation: [#######-----------------------] 50/198 ( 25.3%) | elapsed 35:40 | eta 01:45:35Saved checkpoint at 50/198 to iterative_eval_checkpoint.csv.
Evaluation: [#########---------------------] 60/198 ( 30.3%) | elapsed 42:36 | eta 01:38:00Saved checkpoint at 60/198 to iterative_eval_checkpoint.csv.
Evaluation: [##########----

,topic_id,topic,Baseline,OneStep,Iterative_First,Iterative_Final,Iterative_Steps,Iterative_NumFollowups
0,1,obama family tree,"Was Barack Obama's father, Barack Obama Sr., Kenyan or American?",What is the ancestry of Barack Obama's paternal grandfather?,What is the ancestry of Barack Obama's paternal grandfather?,What other family members are included in Barack Obama's immediate family tree?,3,2
1,10,cheap internet,Is there a specific region or country where you're looking for affordable internet options?,What specific information do you want to know about cheap internet?,What specific information do you want to know about cheap internet?,What specific information do you want to know about cheap internet?,2,1
2,101,ritz carlton lake las vegas,"Is the Ritz-Carlton, Lake Las Vegas located within the city limits of Henderson or on the outskirts?","What is the address of The Ritz-Carlton, Lake Las Vegas?","What is the address of The Ritz-Carlton, Lake Las Vegas?","What is the address of The Ritz-Carlton, Lake Las Vegas?",2,1
3,102,fickle creek farm,Is Fickle Creek Farm located in a specific state or region?,What is the main purpose or type of Fickle Creek Farm?,What is the main purpose or type of Fickle Creek Farm?,What is the main purpose or type of Fickle Creek Farm?,2,1
4,103,madam cj walker,Was Madam C.J. Walker an African American entrepreneur who built her hair care business from scratch in the early 20th century?,What was Madam C.J. Walker's profession?,What was Madam C.J. Walker's profession?,What notable product line did Madam C.J. Walker develop to promote hair care for African American women?,3,2


In [36]:
# attach one reference question per topic
final_df = prepare_final_eval_df(full_df, df)

In [37]:
# remove broken rows only where the reference is missing
print(len(final_df))

198


In [38]:
# Evaluation using BERTScore
# Uses batching so progress stays visible and memory usage is steadier.


In [39]:
refs = final_df['question'].tolist()

metric_results = {
    'Baseline': compute_bertscore_batched(final_df['Baseline'].tolist(), refs, batch_size=64, label='Baseline BERTScore'),
    'OneStep': compute_bertscore_batched(final_df['OneStep'].tolist(), refs, batch_size=64, label='OneStep BERTScore'),
    'Iterative_First': compute_bertscore_batched(final_df['Iterative_First'].tolist(), refs, batch_size=64, label='Iterative_First BERTScore'),
    'Iterative_Final': compute_bertscore_batched(final_df['Iterative_Final'].tolist(), refs, batch_size=64, label='Iterative_Final BERTScore'),
}

for name, value in metric_results.items():
    print(f"{name}: {value:.4f}")

print("Average iterative steps:", round(final_df['Iterative_Steps'].mean(), 3))
print("Average iterative followups:", round(final_df['Iterative_NumFollowups'].mean(), 3))


Baseline BERTScore: [------------------------------] 0/198 (  0.0%) | elapsed 00:00 | eta 00:00

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 1176.87it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline BERTScore: [#########---------------------] 64/198 ( 32.3%) | elapsed 00:17 | eta 00:37

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3318.93it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline BERTScore: [###################-----------] 128/198 ( 64.6%) | elapsed 00:29 | eta 00:16

...

Iterative_Final BERTScore: [###################-----------] 128/198 ( 64.6%) | elapsed 00:14 | eta 00:08

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3343.82it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Iterative_Final BERTScore: [#############################-] 192/198 ( 97.0%) | elapsed 00:24 | eta 00:00

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 3101.03it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Iterative_Final BERTScore: [##############################] 198/198 (100.0%) | elapsed 00:26 | eta 00:00
Baseline: 0.8597
OneStep: 0.8722
Iterative_First: 0.8723
Iterative_Final: 0.8658
Average iterative steps: 2.732
Average iterative followups: 1.732


In [40]:
final_df.to_csv("final_results_iterative_eval.csv", index=False)
print("Saved final results to final_results_iterative_eval.csv")


Saved final results to final_results_iterative_eval.csv


In [41]:
# comparison table

sample_compare = full_df.sample(n=10, random_state=42).copy()

iter1_list = []
iter2_list = []
iter_final_list = []
iter_steps_list = []

for query in sample_compare['topic']:
    res = run_iterative_pipeline(query)
    followups = [step['followup'] for step in res if step.get('followup')]

    iter1_list.append(followups[0] if len(followups) >= 1 else None)
    iter2_list.append(followups[1] if len(followups) >= 2 else None)
    iter_final_list.append(followups[-1] if followups else None)
    iter_steps_list.append(len(res))

sample_compare['Iter-1'] = iter1_list
sample_compare['Iter-2'] = iter2_list
sample_compare['Iter-Final'] = iter_final_list
sample_compare['Iter-Steps'] = iter_steps_list


In [42]:
final_table = sample_compare[['topic', 'Baseline', 'OneStep', 'Iter-1', 'Iter-2', 'Iter-Final', 'Iter-Steps']]

pd.set_option('display.max_colwidth', None)
print(final_table)


                                       topic  \
65                               porterville   
114                              rick warren   
16                       adobe indian houses   
141                          indexed annuity   
156                                 bellevue   
126                       elliptical trainer   
140                alexian brothers hospital   
30                   dutchess county tourism   
18            california franchise tax board   
167  to be or not to be that is the question   

                                                                                                                                                                                                                       Baseline  \
65                                                                                                                                       Is Porterville a specific city or location you're referring to, such as in California?   
114              

In [43]:
final_table.to_csv("comparison_table_iterative.csv", index=False)